# Tensor-hypercontraction walk resources

Compile the complete original-sized walk and inspect its native circuit costs without executing a dense full state.

## Source-scale construction is not source-scale state simulation

The displayed clock/walk factory composes explicit FlagQuantum PREPARE,
SELECT, inverse and clock Circuit blocks. It uses a native synth amplitude
loader for its small clock register, but the complete 83-logical-bit algorithm
is not yet a single high-level qfunc. Host code supplies tensor coefficients,
QROM rows, comparison layout and the source reversible arithmetic.

Inspect logical versus allocated width, resource counts and local truth-table
checks before changing epsilon or spin dimensions. The full program is
compiled and structurally checked; it is not replayed as a 2**83 statevector.
Small-kernel numerical checks must not be presented as full-scale amplitude
validation. A qfunc block-composition contract remains a separate capability
gap, and no large-scale runtime or quantum-advantage claim is made.


Load the native compiler and reproducible inputs.

In [ ]:
from flagquantum import synth
import flagquantum as fq

from examples.synthesis.applications.chemistry.tensor_hypercontraction.overview.quantum import (
    CLOCK_STEPS,
    Layout,
    Leaf,
    Model,
    N_MN,
    append_qft,
    clock_state,
    empty,
    table_walk,
    wide_phase_flip,
)


def full_walk_preview(
    model: Model,
    layout: Layout,
    prepare: fq.Circuit,
    inverse: fq.Circuit,
    select: fq.Circuit,
) -> fq.Circuit:
    def leaf(with_select: bool) -> Leaf:
        def walk(circuit: fq.Circuit, control: int) -> None:
            if with_select:
                circuit.compose(select)
            circuit.compose(inverse)
            for wire in (*layout.ell, control):
                circuit.x(wire)
            first, second = layout.comparison[:2]
            wide_phase_flip(circuit, layout.ell, control, (first, second))
            for wire in (*layout.ell, control):
                circuit.x(wire)
            circuit.compose(prepare)

        return walk

    complete = empty(layout)
    initialized = synth.synthesize(
        clock_state,
        classical_inputs={"weights": list(model.clock)},
        max_qubits=4,
        max_workspace=0,
        memory_limit_bytes=1024**2,
    )
    complete.compose(initialized.circuit, qubits=layout.clock[::-1])
    complete.h(layout.plus1).h(layout.plus2)
    table_walk(
        complete,
        layout.clock,
        (leaf(False),) + (leaf(True),) * (CLOCK_STEPS - 1),
        layout.path[N_MN:],
    )
    append_qft(complete, layout.clock[::-1], inverse=True)
    return complete


from examples.synthesis.applications.chemistry.tensor_hypercontraction.overview import (
    main,
    model,
    quantum,
)
from examples.synthesis.applications.chemistry.tensor_hypercontraction.overview.layout import (
    allocate_layout,
)


Keep original precision and spin-register widths.

In [ ]:
assert (model.N_EPSILON, model.N_UD) == (4, 3)


Generate all tables, their rotation, and the sine clock.

In [ ]:
data = model.generate()
main.describe(data)


Allocate the full record, spin, and clock layout.

In [ ]:
layout = allocate_layout()
assert layout.logical_width == 83


Reserve the traversal flags for nested clock and QROM calls.

In [ ]:
assert len(layout.path) == model.N_MN + model.N_EPSILON


Compile the first PREPARE block.

In [ ]:
early = quantum.equal_superposition(data, layout)


Use native IR resource reporting.

In [ ]:
report = main.report


Inspect the resources for PREPARE part 1.

In [ ]:
report("PREPARE part 1", early)


Compile address arithmetic, alias lookup and swaps.

In [ ]:
late = quantum.finish_prepare(data, layout)


Inspect the resources for PREPARE part 2.

In [ ]:
report("PREPARE part 2", late)


Combine both PREPARE stages.

In [ ]:
prepare = quantum.empty(layout).compose(early).compose(late)


Compile SELECT, including digital rotations and phases.

In [ ]:
select = quantum.select_block(data, layout)


Inspect SELECT resources.

In [ ]:
report("SELECT", select)


Use the inverse preparation and a wide controlled phase for reflection.

In [ ]:
inverse = prepare.adjoint()


Add the original clock and spin state preparations and the inverse QFT.

In [ ]:
complete = full_walk_preview(data, layout, prepare, inverse, select)


Materialize the full native circuit product.

In [ ]:
assert complete.to_ir().n_wires == layout.width


Report the full resources and audit the emitted reversible subcircuits.

In [ ]:
report("Full clock program", complete)
main.report_checks(data, layout)
